# Rosenbrock — Discovering η₂ = θ₂ + θ₁²

Toy two-parameter inference task with a known **nonlinear** degeneracy.
PCA of the Fisher matrix cannot recover it; the Distillery does.

* **Model**: $x \sim \mathcal{N}(\boldsymbol{\mu}_x(\boldsymbol{\theta}), \Sigma)$ with
  $\boldsymbol{\mu}_x = (\theta_1,\, \theta_2 - \theta_1^2)^\top$ and
  $\Sigma = \mathrm{diag}(1, 4)$.
* **Data**: $n_d = 50$ i.i.d. draws from that distribution, flattened into
  one $100$-vector.
* **Ground truth**: $\eta_2 = \theta_2 + \theta_1^2$.


In [ ]:
# Colab setup
!git clone https://github.com/tlmakinen/degeneracy_distillery.git
%cd /content/degeneracy_distillery
!pip install -q -e .
%cd /content/
!git clone https://github.com/DeaglanBartlett/ESR.git
%cd /content/ESR
!pip install -q -e .


**Restart the runtime** before continuing on Colab.

In [ ]:
# Colab only: uncomment to hard-restart the runtime after the install above.
# Skip on a local Jupyter kernel — this will kill your session.
# import os
# os.kill(os.getpid(), 9)


In [ ]:
import esr.generation.generator
import numpy as np
import jax, jax.numpy as jnp, jax.random as jr
import matplotlib.pyplot as plt
import sympy

from degeneracy_distillery.training_loop_fishnets import train_fishnets
from degeneracy_distillery.training_loop_flatten import fit_flattening
from degeneracy_distillery.align_coords import load_and_process_data_v2
from degeneracy_distillery.sr_utils import (
    fit_and_analyze_sr, analyze_equations, sr_structure_predicate,
    check_symbolic_invertibility,
    fit_theta_scaler, expressions_to_physical,
)
from degeneracy_distillery.postprocess_new import (
    analyze_atom_sharing, regroup_like_terms,
)
from degeneracy_distillery.postprocessing_utils import (
    print_discovered_expressions, get_y_sr,
    flatten_with_numerical_jacobian, check_flattening,
)
from degeneracy_distillery.preprocessing_utils import get_eigenvalues

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'figure.figsize': (8, 5),
    'figure.dpi': 130, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})


## 1. Simulator and data generation

In [ ]:
key = jr.PRNGKey(0)
dim, n_d = 2, 50
nsims = 250
MIN_MU, MAX_MU = -3.0, 3.0
Sigma = jnp.diag(jnp.array([1.0, 2.0]) ** 2)


def simulator(rng, theta):
    """x_mean = (theta1, theta2 - theta1**2); n_d i.i.d. draws."""
    x_mean = jnp.array([theta[0], theta[1] - theta[0] ** 2])
    return jr.multivariate_normal(rng, mean=x_mean, cov=Sigma, shape=(n_d,)).reshape(-1)


k1, k2 = jr.split(key)
theta_train = np.asarray(jr.uniform(k1, (nsims, dim), minval=MIN_MU, maxval=MAX_MU))
data_train  = np.asarray(jax.vmap(simulator)(jr.split(k1, nsims), theta_train))

theta_test = np.asarray(jr.uniform(k2, (nsims, dim), minval=MIN_MU, maxval=MAX_MU))
data_test  = np.asarray(jax.vmap(simulator)(jr.split(k2, nsims), theta_test))
print("theta_train", theta_train.shape, "data_train", data_train.shape)


## 2. Pre-fishnet rescaling

θ already lives in $[-3, 3]^2$, but applying the same `(1, 2)` scaling
keeps the convention shared across the tutorial notebooks and gives a
clean affine inverse for the final symbolic substitution.

In [ ]:
scaler = fit_theta_scaler(theta_train, feature_range=(-3.0, 3.0))
theta_train_s = scaler.transform(theta_train).astype(np.float32)
theta_test_s  = scaler.transform(theta_test).astype(np.float32)
print("scaled range:", theta_train_s.min(0), theta_train_s.max(0))


## 3. Fisher-network ensemble

In [ ]:
_ = train_fishnets(
    theta_train_s, data_train,
    theta_test_s,  data_test,
    num_models=20,
    hids_min=10, hids_max=300, n_layers=[2, 5],
    train_epochs=1000, train_min_epochs=100, patience=20,
    train_batch_size=25, lr=5e-5,
    seed_model=201, seed_train=999,
    outdir="fishnets-rosen",
)


## 4. Flattening normalising flow

In [ ]:
fish = np.load("fishnets-rosen/fishnets_outputs.npz")
thetas = jnp.array(fish["theta"])
ensemble_weights = fish["ensemble_weights"]
F_network_ensemble = jnp.array(fish["Fs"])

w, ensemble_w, outputs_flatten, flatten_model = fit_flattening(
    F_network_ensemble, thetas,
    ensemble_weights=ensemble_weights,
    hidden_size=256, n_layers=7,
    batch_size=50,
    epochs_phase1=1000, epochs_phase2=2000, finetune_epochs=250,
    min_epochs=250, patience=40,
    lr_phase1=1e-6, lr_schedule_initial=7e-5, lr_decay=0.3, lr_finetune=4e-6,
    Fisher_to_flatten="average",
    norm_factor=None, norm_method="median_det",
    flattener_activation="softplus",
    noise=1e-4, seed=0,
    output_prefix="rosen_flatten",
    use_whitening=True, nn_inv=False,
    forward_backward_mlp=True,
    l1_alpha=0.0, do_plot=False,
    return_model=True
)


## 5. Coordinate alignment

In [ ]:
data = load_and_process_data_v2(
    datapath="./",
    filename="rosen_flatten.npz",
    num_samps=4000, seed=44,
    process_ensemble=True, n_d=1.0,
    align_mode="procrustes",
    separate_nonlinearity=True,
    canonicalize="sign_only",
    use_prior_normalization=True,
    restore_reference_mean=False,
    Fisher_to_flatten="average",
    verbose=False,
)

X = data["X"]
X, y, y_std, dy_sr, Fs = X, data["y"], data["y_std"], data["dy_sr"], data["Fs"]
min_ = y.min(0)
y -= min_
ys = data["ys"] - min_
n_params = X.shape[1]
print("aligned X", X.shape, "y", y.shape)


In [ ]:
from degeneracy_distillery.postprocessing_utils import weighted_std

X_sr = jr.uniform(key, minval=X.min(0), maxval=X.max(0), shape=(2000, 2))

ys_sr = jnp.array([jax.vmap(lambda x: flatten_model.apply(w_i, x))(X_sr) for w_i in ensemble_w])

ys_sr_rot = np.array([
    np.einsum("ij,bj->bi", data['rotmats'][i], ys_sr[i] - ys_sr[i].mean(0))
    for i in range(len(ys_sr))
])

y_std_sr = weighted_std(ys_sr_rot, data['ensemble_weights'])
y_sr = np.average(ys_sr_rot, 0, data['ensemble_weights'])
ys_sr_rot -= y_sr.min(0)
y_sr -= y_sr.min(0)

In [ ]:
# Quick sanity check: each aligned y component vs the SR-grid X coordinates.
fig, axs = plt.subplots(1, n_params, figsize=(8, 3.5), sharey=True)
for j, ax in enumerate(axs):
    ax.scatter(X[:, j],    y[:, j],    s=4, alpha=0.5, label="aligned (post-rot)")
    ax.scatter(X_sr[:, j], y_sr[:, j], s=4, alpha=0.3, label="SR grid sample")
    ax.set_xlabel(rf"$X_{j+1}$")
    ax.set_ylabel(rf"$y_{j+1}$" if j == 0 else "")
axs[0].legend(loc="best", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# check coordinate sampling distribution

fig,axs = plt.subplots(1,n_params)
axs = axs.flatten()

bins = [-5, 5]

for i,ax in enumerate(axs):
    ax.hist(((ys[:, :, ...] - y) / y_std)[:, :, i].flatten(), density=True)
    ax.hist(np.random.normal(size=(2000,)).flatten(), histtype='step', density=True)

    xpdf = np.linspace(bins[0], bins[-1], 400)
    ypdf = np.exp(-0.5 * xpdf**2) / np.sqrt(2.0 * np.pi)
    ax.plot(xpdf, ypdf, lw=1.0, c="k", ls="--", label="Normal")

    # ax.set_xlim(-10, 10)
plt.tight_layout()
plt.show()


fig,axs = plt.subplots(1,n_params)
axs = axs.flatten()

bins = [-5, 5]

for i,ax in enumerate(axs):
    ax.hist(((ys_sr_rot - y_sr) / y_std_sr)[:, :, i].flatten(), density=True)
    ax.hist(np.random.normal(size=(2000,)).flatten(), histtype='step', density=True)

    xpdf = np.linspace(bins[0], bins[-1], 400)
    ypdf = np.exp(-0.5 * xpdf**2) / np.sqrt(2.0 * np.pi)
    ax.plot(xpdf, ypdf, lw=1.0, c="k", ls="--", label="Normal")

    # ax.set_xlim(-10, 10)
plt.tight_layout()
plt.show()

In [ ]:
# create scatterplot to see how y responds to X

fig, axs = plt.subplots(n_params, n_params, figsize=(5,5))

skip=10

for i in range(X_sr.shape[-1]):
    for j in range(y_sr.shape[-1]):

        # row, column
        axs[i,j].errorbar(X_sr[::skip,i], y_sr[::skip, j], yerr=y_std_sr[::skip, j], label='', fmt='o', markersize=1)

        axs[i,j].set_xlabel(r"$X_%d$"%(i))
        axs[i,j].set_ylabel(r"$y_%d$"%(j))

plt.tight_layout()
plt.show()

## 6. Symbolic regression

In [ ]:
from degeneracy_distillery.sr_utils import fit_symbolic_regression

fit_symbolic_regression(
    X_sr, y_sr, y_std_sr,
    parent_dir='./sr_results_rosen/',
    random_state=32134,
    time_limit=60*5,
    max_length=25,
    max_depth=10,
    allowed_symbols='add,mul,div,pow,constant,variable,square',
    )

In [ ]:
from degeneracy_distillery.sr_utils import analyze_equations, sr_structure_predicate

X_test, y_test, y_std_test = X, y, y_std
dy_sr_test, Fs_test = dy_sr, Fs

mdl_coords, frob_coords, analysis = analyze_equations(
    X_test, y_test, y_std_test, dy_sr_test, Fs_test,
    parent_dir="sr_results_rosen/",
    n_params=n_params,
    equation_set="pareto",
    max_complexity_thresh=15,
    length_penalty=3.0,
    equation_predicate=sr_structure_predicate(
        n_params=2,
        forbid_self_transcendental=True,
    ),
)
print_discovered_expressions([sympy.simplify(e).evalf(2) for e in mdl_coords])


## 7. Postprocessing (`postprocess_new`)

In [ ]:
report = analyze_atom_sharing(mdl_coords)
pruned_exprs, R, info = regroup_like_terms(
    mdl_coords, X=X_test, Fs=Fs_test, n_params=n_params,
    method="atoms",
    do_snap=True, snap_rel_tol=0.5, snap_flat_tol=0.5,
    decimal=2, threshold=2.0,
)
print_discovered_expressions([sympy.simplify(e).evalf(2) for e in pruned_exprs])
# inv = check_symbolic_invertibility(pruned_exprs, verbose=True)
# print("inverse coords:", inv["inv_coords"])



## 8. Back to physical θ

The flattened-frame X coords are in `[1, 2]`; substituting
$X_i = a_i \theta_i + b_i$ via `expressions_to_physical` recovers the
discovered formulas in $(\theta_1, \theta_2)$.  We expect one component
proportional to $\theta_2 + \theta_1^2$.

In [ ]:
physical_exprs = expressions_to_physical(
    pruned_exprs, scaler,
    sr_offset=0.0,                 # this notebook does not +1 the SR inputs
    theta_names=("theta1", "theta2"),
    decimal=3,
)
for k, e in enumerate(physical_exprs):
    print(f"  eta_{k} = {e}")


## 9. Validation: flatness eigenvalues

A successful flattening drives the eigenvalues of the network Fisher in the
learned coordinates toward unity.

In [ ]:
nn_flats = jax.vmap(flatten_with_numerical_jacobian)(dy_sr_test, Fs_test)
mdl_flats, _    = check_flattening(mdl_coords,    X=X_test, Fs=Fs_test)
pruned_flats, _ = check_flattening(pruned_exprs,  X=X_test, Fs=Fs_test)


def fro_score(Q):
    return np.linalg.norm(np.asarray(Q) - np.eye(n_params), axis=(-2, -1))


for name, Q in [("raw θ", Fs_test),
                ("MDL",   mdl_flats),
                ("pruned", pruned_flats),
                ("NN",    nn_flats)]:
    print(f"  {name:8s}  median ||Q-I||_F = {np.median(fro_score(Q)):.3f}")

evalues_nn     = jax.vmap(get_eigenvalues)(nn_flats)
evalues_pruned = jax.vmap(get_eigenvalues)(pruned_flats)

bins = np.linspace(0, 5, 25)
plt.figure(figsize=(6, 3))
plt.hist(np.array(evalues_nn).flatten(),     bins=bins, alpha=0.55, label="NN")
plt.hist(np.array(evalues_pruned).flatten(), bins=bins, alpha=0.55, label="pruned SR")
plt.axvline(1.0, ls="--", color="k", lw=0.8)
plt.xlabel(r"$\lambda(F_\eta)$"); plt.ylabel("count"); plt.legend()
plt.tight_layout()
plt.show()


## 10. Save artifacts

In [ ]:
import pickle, os
os.makedirs("sr_results_rosen", exist_ok=True)
with open("sr_results_rosen/sr_expressions.pkl", "wb") as f:
    pickle.dump({
        "mdl_coords":      mdl_coords,
        "frob_coords":     frob_coords,
        "pruned_exprs":    pruned_exprs,
        "physical_exprs":  [str(e) for e in physical_exprs],
        "scaler_scale":    scaler.scale_,
        "scaler_min":      scaler.min_,
        "scaler_data_min": scaler.data_min_,
        "scaler_data_max": scaler.data_max_,
    }, f)
print("saved sr_results_rosen/sr_expressions.pkl")


In [ ]:
!cp -r fishnets-rosen/ sr_results_rosen/
!cp rosen_flatten.npz sr_results_rosen/
!zip -r sr_results_rosen_tiny.zip sr_results_rosen/